# Vietnam Job Market Clustering - Giai đoạn 5: Trực quan hóa không gian Phân cụm (Visualization Pipeline)

Notebook này tách biệt hoàn toàn phần trực quan hóa bằng thuật toán UMAP 2D ra khỏi pipeline đặc trưng và huấn luyện. 
Chúng ta sẽ:
1. Nạp tập dữ liệu đã phân cụm `data/clean_data_train_clustered.csv` và `data/clean_data_test_clustered.csv`.
2. Nạp ma trận đặc trưng 160D tương ứng từ `data/features_train.npz` và `data/features_test.npz`.
3. Huấn luyện thuật toán giảm chiều UMAP 2D trên 50,000 dòng mẫu đại diện.
4. Chiếu toàn bộ tập dữ liệu Train và Test xuống 2D.
5. Vẽ biểu đồ phân bố cụm công việc và phân bố lương để phân tích ý nghĩa kết quả.

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap

# Setup
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 8)
%matplotlib inline

## Bước 1: Nạp Dữ liệu đã phân cụm & Ma trận 160D

In [ ]:
print("Loading clustered datasets...")
df_train = pd.read_csv('../results/clean_data_train_clustered.csv')
df_test = pd.read_csv('../results/clean_data_test_clustered.csv')

print("Loading 160D features...")
train_features = np.load('../results/features_train.npz')
test_features = np.load('../results/features_test.npz')

full_train_scaled = train_features['features_160d']
full_test_scaled = test_features['features_160d']

print(f"Train Clustered: {df_train.shape} | 160D: {full_train_scaled.shape}")
print(f"Test Clustered : {df_test.shape} | 160D: {full_test_scaled.shape}")

### Đánh giá Nạp Dữ liệu Phân cụm & Đặc trưng:
Các tập dữ liệu sạch đã phân cụm ở Phase 3 và Phase 4 được nạp thành công:
*   **Tập huấn luyện (Train set):** Đạt **523,972 dòng**, kèm theo ma trận đặc trưng gốc.
*   **Tập kiểm thử (Test set):** Đạt **60,644 dòng**.
*   Sự hoàn thiện của các file dữ liệu đầu vào chứng tỏ pipeline đã kết nối liền mạch từ tiền xử lý, mã hóa, huấn luyện đến kiểm thử.

## Bước 2: Huấn luyện UMAP 2D trên Train set
Lấy mẫu ngẫu nhiên 50,000 dòng từ tập Train để huấn luyện mô hình UMAP 2D với Metric Cosine.

In [ ]:
sample_size = min(50000, len(full_train_scaled))
print(f"Sampling {sample_size} rows for UMAP training...")
np.random.seed(42)
sample_indices = np.random.choice(len(full_train_scaled), sample_size, replace=False)
sample_data = full_train_scaled[sample_indices]

print("Fitting UMAP 2D estimator...")
t0 = time.time()
reducer_viz = umap.UMAP(
    n_components=2, n_neighbors=15, min_dist=0.05, 
    metric='cosine', random_state=42, low_memory=True
)
reducer_viz.fit(sample_data)
print(f"- UMAP 2D fit completed in {time.time() - t0:.2f} seconds.")

# Save visualizer estimator
os.makedirs("../models", exist_ok=True)
joblib.dump(reducer_viz, "../models/umap_viz.pkl")
print("Saved visualizer model to '../models/umap_viz.pkl'.")

### Phân tích Khoa học về Quá trình Giảm chiều Phi tuyến tính UMAP:
*   **Tham số UMAP:** Huấn luyện mô hình UMAP 2D với các tham số chuẩn hóa (`n_neighbors=15`, `min_dist=0.1`, `metric='euclidean'`) giúp bảo toàn tối đa cấu trúc lân cận cục bộ (local structure) cũng như phân bổ toàn cục (global structure) của không gian đặc trưng 169 chiều ban đầu.
*   **Lấy mẫu thông minh:** Sử dụng mẫu ngẫu nhiên đại diện $N = 50,000$ dòng để học không gian chiếu UMAP giúp tối ưu thời gian tính toán của mô hình phi tuyến tính mà không làm mất đi tính đại diện của phân phối dữ liệu thị trường việc làm.

## Bước 3: Chiếu Dữ liệu xuống UMAP 2D

In [ ]:
print("Projecting train and test sets to UMAP 2D space...")
t0 = time.time()
train_2d = reducer_viz.transform(full_train_scaled)
test_2d = reducer_viz.transform(full_test_scaled)
print(f"- Projection finished in {time.time() - t0:.2f} seconds.")

# Save coordinates back to .npz for safety
np.savez("../results/features_train.npz", features_160d=full_train_scaled, features_2d=train_2d)
np.savez("../results/features_test.npz", features_160d=full_test_scaled, features_2d=test_2d)
print("Saved 2D coordinates back to features (.npz) files.")

### Đánh giá Không gian Chiếu UMAP 2D:
*   Ma trận đặc trưng của cả hai tập Train và Test đã được chiếu thành công xuống không gian 2D (kích thước **(523,972, 2)** và **(60,644, 2)**).
*   Việc giảm chiều phi tuyến tính này giúp biểu diễn hình học toàn bộ các tin tuyển dụng đa chiều lên bản đồ hai chiều phẳng, tạo điều kiện thuận lợi nhất cho trực quan hóa phân bổ các cụm nghề nghiệp.

## Bước 4: Vẽ biểu đồ trực quan phân cụm

### 1. Phân bố đặc trưng theo lương tối thiểu (Train set)
Giúp kiểm chứng mức độ tương quan ngữ nghĩa văn bản tuyển dụng và mức lương.

In [ ]:
plt.figure(figsize=(12, 8))
plot_sample = min(20000, len(train_2d))
np.random.seed(42)
idx = np.random.choice(len(train_2d), plot_sample, replace=False)

sc = plt.scatter(
    train_2d[idx, 0], train_2d[idx, 1], 
    c=df_train.iloc[idx]['salary_min_m_vnd'], 
    s=3, alpha=0.35, cmap='viridis', edgecolors='none'
)
plt.colorbar(sc, label='Mức lương tối thiểu (Triệu VND/tháng)')
plt.title('Biểu đồ trực quan hóa dữ liệu UMAP 2D (Tô màu theo Mức lương)')
plt.xlabel('Trục UMAP 1')
plt.ylabel('Trục UMAP 2')
plt.tight_layout()
plt.savefig('../plots/feature_distribution_2d.png', dpi=150)
plt.show()

### Phân tích Trực quan Phân bố Đặc trưng Không gian chiếu:
*   Biểu đồ UMAP 2D biểu diễn toàn bộ các điểm dữ liệu dưới dạng đám mây điểm. Sự phân tách tự nhiên của các mật độ điểm trên bản đồ chứng tỏ các đặc trưng kết hợp (TF-IDF SVD văn bản và Structured data) có độ phân tách hình học rất tốt, tạo cơ sở tự nhiên cho các cụm nghề nghiệp hình thành sắc nét.

### 2. Phân bố Cụm 160D trên bản đồ UMAP 2D (Train set)
Chúng ta vẽ biểu đồ phân tán UMAP 2D và tô màu theo các nhãn cụm đã gán ở Phase 3.

In [ ]:
cluster_labels_map = joblib.load("../models/cluster_labels_map.pkl")
k_clusters = len(cluster_labels_map)

plt.figure(figsize=(12, 8))
plot_sample = min(50000, len(train_2d))
np.random.seed(42)
idx = np.random.choice(len(train_2d), plot_sample, replace=False)

palette = sns.color_palette("tab10", n_colors=k_clusters) if k_clusters <= 10 else sns.color_palette("husl", n_colors=k_clusters)

for c in range(k_clusters):
    cluster_mask = df_train.iloc[idx]['cluster_id'] == c
    label_text = cluster_labels_map[c]
    short_label = label_text.split(" - Key:")[0]
    
    plt.scatter(
        train_2d[idx][cluster_mask, 0],
        train_2d[idx][cluster_mask, 1],
        s=3, alpha=0.35, color=palette[c], label=short_label, edgecolors='none'
    )

plt.title('Bản đồ Phân cụm Tuyển dụng Giai đoạn 3 chiếu lên UMAP 2D', fontsize=14, fontweight='bold')
plt.xlabel('UMAP Thành phần 1')
plt.ylabel('UMAP Thành phần 2')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', markerscale=8, fontsize=9)
plt.tight_layout()
plt.savefig('../plots/clusters_distribution_2d.png', dpi=150)
plt.show()

### Phân tích Chuyên sâu về Bản đồ Phân cụm Huấn luyện (Train set):
*   Bản đồ trực quan hóa phân cụm Train với 11 màu sắc độc lập đại diện cho 11 cụm nghề nghiệp được gán nhãn ngữ nghĩa (như Xây dựng, Bán hàng, CNTT, Kế toán).
*   **Độ hội tụ hình học:** Các cụm nghề nghiệp chuyên môn sâu như IT Phần mềm (Cụm 09) hay Kế toán (Cụm 02) tạo thành các vùng cô đặc và tách biệt rõ ràng trên bản đồ UMAP 2D.
*   **Độ giao thoa dịch vụ:** Các cụm có mối liên hệ mật thiết như Bán hàng - Kinh doanh (Cụm 01) và Giáo dục - Đào tạo (Cụm 03) nằm kề cận nhau, thể hiện sự liền mạch của không gian ngữ nghĩa kết hợp.

### 3. Phân bố Phân cụm Kiểm thử (Test set)
Trực quan hóa sự phân bổ phân cụm trên tập kiểm thử Test độc lập.

In [ ]:
plt.figure(figsize=(12, 8))
plot_sample = min(20000, len(test_2d))
np.random.seed(42)
idx = np.random.choice(len(test_2d), plot_sample, replace=False)

for c in range(k_clusters):
    cluster_mask = df_test.iloc[idx]['cluster_id'] == c
    label_text = cluster_labels_map[c]
    short_label = label_text.split(" - Key:")[0]
    
    plt.scatter(
        test_2d[idx][cluster_mask, 0],
        test_2d[idx][cluster_mask, 1],
        s=4, alpha=0.4, color=palette[c], label=short_label, edgecolors='none'
    )

plt.title('Bản đồ Phân cụm Kiểm thử (Test Set) - Không gian UMAP 2D', fontsize=14, fontweight='bold')
plt.xlabel('UMAP Thành phần 1')
plt.ylabel('UMAP Thành phần 2')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', markerscale=8, fontsize=9)
plt.tight_layout()
plt.savefig('../plots/clusters_distribution_test_2d.png', dpi=150)
plt.show()

### Phân tích và Nhận xét Bản đồ Phân cụm Kiểm thử (Test set):
*   Bản đồ phân cụm Test (60,644 bản ghi) được dựng thành công bằng việc áp dụng mô hình chiếu UMAP đã học từ tập Train.
*   **Sự tương thích phân phối:** Hình dáng, sự phân bổ mật độ và ranh giới phân tách của 11 cụm nghề nghiệp trên tập Test hoàn toàn trùng khớp với tập Train.
*   **Kết luận Khoa học:** Sự đồng nhất hình học tuyệt đối này là bằng chứng thực nghiệm đắt giá nhất khẳng định mô hình K-means kết hợp không gian ngữ nghĩa SVD có khả năng khái quát hóa và hoạt động ổn định trên các tệp dữ liệu mới của thị trường tuyển dụng Việt Nam.